<div align="center">
  <img src="https://datasupplychain.waverlyrose.io/healthstack-logo-dark.png" alt="HealthStack logo" width="320" />
</div>

<p align="center"><strong>AFROTECH 2026</strong></p>
<p align="center"><font size="6"><strong>Building Fairness Into the Data Supply Chain</strong></font></p>
<p align="center"><strong>Instructor: Waverly Rose Brim</strong></p>

---

<p align="center"><font size="6"><strong>Machine Learning Missteps</strong></font></p>

<div align="center">
  <img src="https://datasupplychain.waverlyrose.io/ml-missteps.jpg" alt="A clinician in a white coat and blue gloves interacts with a holographic medical display of a human figure." width="640" />
  <p><em>A single engineering decision can redraw the boundaries of diagnosis.</em></p>
</div>

---

## Start here
This notebook answers three questions:
1. What are the different operations people call "removing race"?
2. Do deleting the coefficient and refitting the equation move the same people?
3. Why do you have to read both directions of movement?

**Workflow:** predict -> run -> inspect -> interpret.

## Colab setup
Skip this cell if you are running locally.

In [ ]:
import sys, subprocess

if "google.colab" in sys.modules:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade", "--force-reinstall", "--no-deps", "--no-cache-dir",
        "data-supply-chain-workshop==0.1.2",
    ])
else:
    import data_supply_chain

## Cold start
Same cohort, same split, whichever notebook you opened first. Earlier
notebooks do not need to be open.

In [ ]:
from data_supply_chain.bootstrap import cold_start

S = cold_start()
print(f"Cohort loaded: {len(S.df):,} adults · source={S.source} · real={S.is_real}")

## **"Remove race" is not one operation**
There are at least three different operations:

- A: **Delete the 1.159 coefficient** from the existing equation.
- B: **Refit** a new equation without race (CKD-EPI 2021).
- C: **Remove race from a model** while keeping race-encoded features.

This step compares A vs B. 

Strategy C is step 5 (`05_Model_Is_Blind`): a model with no race column that still consumes race-built eGFR.

In [ ]:
# Demo, take this opportunity to review the output

from data_supply_chain.egfr import add_egfr_columns, egfr_asr_nb, removal_rates

df = add_egfr_columns(S.df)
df["egfr_asr_nb"] = egfr_asr_nb(df["egfr_2009"], df["black"])

removals = {
    "A - delete the coefficient": removal_rates(df, "egfr_asr_nb"),
    "B - refit (CKD-EPI 2021)": removal_rates(df, "egfr_2021"),
}

print(f"{'':30s}{'INTO CKD':>12s}{'OUT OF CKD':>13s}")
print("-" * 55)
for name, r in removals.items():
    print(f"  {name}")
    for grp, key in (("Black", "black"), ("Non-Black", "nonblack")):
        print(f"    {grp:24s}{r[key + '_into']:10.2f}%{r[key + '_out']:12.2f}%")
    print()

## Interpret the result

In [ ]:
_ratio = removals["A - delete the coefficient"]["black_into"] / removals["B - refit (CKD-EPI 2021)"]["black_into"]
print(f"1) Deleting the coefficient reclassifies {_ratio:.2f}x as many Black adults into CKD as the refit.")
print("2) Non-Black adults are not untouched by the refit; movement goes the other way:")
print(
    f"   {removals['B - refit (CKD-EPI 2021)']['nonblack_into']:.2f}% into CKD, "
    f"{removals['B - refit (CKD-EPI 2021)']['nonblack_out']:.2f}% out of CKD."
)
print("3) The blood test did not change. The transformation did.")

In essence, 

**1) A is more aggressive than B.** Deleting the coefficient moves more Black adults into CKD than the 2021 refit. Both get called "removing race." They do not move the same people.

**2) Read both directions.** Under the refit, Non-Black adults do not move into CKD. They move *out*. "0.00% reclassified" can be true for one direction and still miss everyone who changed category the other way.

**3) The blood test did not change.** Same creatinine. The transformation did. That is why two operations that sound like the same fairness fix produce different diagnoses.

In [ ]:
from IPython.display import Image, clear_output, display
from data_supply_chain.figures import reclass_png

clear_output(wait=True)
display(Image(data=reclass_png(removals), format="png"))

---

## Maintainer cell (safe to skip in workshop)
This writes the findings the website reads, so the numbers on the page stay
synced to this notebook's output.

In [ ]:
from data_supply_chain.findings import Findings
from data_supply_chain.paths import findings_path

f = Findings.open()
for _key, _name in (("drop", "A - delete the coefficient"), ("refit", "B - refit (CKD-EPI 2021)")):
    for _grp in ("black", "nonblack"):
        for _dir in ("into", "out"):
            f.record(
                f"act1.removals.{_key}.{_grp}_{_dir}",
                round(removals[_name][f"{_grp}_{_dir}"], 2),
            )
f.record("act1.removals.drop_vs_refit_ratio", round(_ratio, 2))

if findings_path() is None:
    print("Outside a repo checkout - findings.json not written (expected in Colab).")
else:
    f.write(state=S)
    print("wrote findings.json")